# ForgeLM Quick Start — Supervised Fine-Tuning (SFT)

Fine-tune a language model on your custom dataset using ForgeLM.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/quickstart_sft.ipynb)

In [ ]:
# Install ForgeLM (clean install to avoid Colab dependency conflicts)
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git
!pip uninstall -y wandb -q  # Remove conflicting wandb (not needed, tensorboard is default)

In [ ]:
# Create a training config
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 512
  load_in_4bit: false
  backend: "transformers"

lora:
  r: 16
  alpha: 32
  dropout: 0.1
  target_modules:
    - "q_proj"
    - "v_proj"

training:
  output_dir: "./checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 2.0e-5
  eval_steps: 50
  save_steps: 50
  save_total_limit: 2

data:
  dataset_name_or_path: "timdettmers/openassistant-guanaco"
  shuffle: true
"""

with open("my_config.yaml", "w") as f:
    f.write(config_yaml)

print("Config saved!")

In [ ]:
# Validate config (dry run)
!forgelm --config my_config.yaml --dry-run

In [ ]:
# Start training!
!forgelm --config my_config.yaml

In [ ]:
# Load and test the fine-tuned model
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")
model = PeftModel.from_pretrained(base_model, "./checkpoints/final_model")
tokenizer = AutoTokenizer.from_pretrained("./checkpoints/final_model")

inputs = tokenizer("What is machine learning?", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))